# Real World Dataset Analysis

using dataset "bob_ross_paintings.csv"

from https://github.com/jwilber/Bob_Ross_Paintings

In [294]:
# Import the necessary libraries and tools

import pandas as pd
import ast
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import seaborn as sns

In [295]:
# Read in the data
br_paint = pd.read_csv('bob_ross_paintings.csv')

In [296]:
# See info about each column
br_paint.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 403 entries, 0 to 402
Data columns (total 29 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        403 non-null    int64 
 1   painting_index    403 non-null    int64 
 2   img_src           403 non-null    object
 3   painting_title    403 non-null    object
 4   season            403 non-null    int64 
 5   episode           403 non-null    int64 
 6   num_colors        403 non-null    int64 
 7   youtube_src       403 non-null    object
 8   colors            403 non-null    object
 9   color_hex         403 non-null    object
 10  tags              403 non-null    object
 11  Black_Gesso       403 non-null    int64 
 12  Bright_Red        403 non-null    int64 
 13  Burnt_Umber       403 non-null    int64 
 14  Cadmium_Yellow    403 non-null    int64 
 15  Dark_Sienna       403 non-null    int64 
 16  Indian_Red        403 non-null    int64 
 17  Indian_Yellow   

In [297]:
# See how many columns and rows in the dataset
br_paint.shape

(403, 29)

In [298]:
# See the first 5 rows
br_paint.head()

In [299]:
# See the last 4 rows
br_paint.tail(4)

In [300]:
# get stats
br_paint.describe().T

In [306]:
# How many seasons of Bob Ross Paintings exist
total_seasons = br_paint["season"].drop_duplicates()
total_seasons.count()
print("Total # of Seasons:", total_seasons.count())

Total # of Seasons: 31


In [307]:
# How many episodes of Bob Ross Paintings exist
episode_counts = br_paint["episode"]
print("Total # of Episodes:", episode_counts.count())

Total # of Episodes: 403


# 🔥 HEATMAP GALLERY

5 Amazing Heatmaps You Can Make From This Dataset!

In [ ]:
# HEATMAP #1: Color Usage by Season
# Shows which colors were used more in early vs late seasons

color_columns = [
    'Black_Gesso', 'Bright_Red', 'Burnt_Umber', 'Cadmium_Yellow', 
    'Dark_Sienna', 'Indian_Yellow', 'Liquid_Black', 'Liquid_Clear', 
    'Midnight_Black', 'Phthalo_Blue', 'Phthalo_Green', 'Prussian_Blue', 
    'Sap_Green', 'Titanium_White', 'Van_Dyke_Brown', 'Yellow_Ochre', 
    'Alizarin_Crimson'
]

# Calculate average color usage per season
season_color_usage = br_paint.groupby('season')[color_columns].mean()

plt.figure(figsize=(16, 6))
sns.heatmap(season_color_usage, annot=False, cmap='viridis', 
            cbar_kws={'label': 'Frequency of Use'},
            linewidths=0.5)
plt.title('🎨 Color Usage Evolution Across 31 Seasons', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Colors', fontsize=12)
plt.ylabel('Season', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("What you see: Darker = used more frequently in that season")

In [ ]:
# HEATMAP #2: Color Correlation Matrix
# Shows which colors are used together (correlation between colors)

correlation_matrix = br_paint[color_columns].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'label': 'Correlation'})
plt.title('🎨 Which Colors Are Used Together? (Color Correlation)', 
         fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("What you see: Red = highly correlated (used together), Blue = inversely correlated")

In [ ]:
# HEATMAP #3: Average # of Colors Used by Season & Episode
# Shows painting complexity (number of colors) across seasons and episodes

season_episode_colors = br_paint.pivot_table(
    values='num_colors', 
    index='season', 
    columns='episode', 
    aggfunc='mean'
)

plt.figure(figsize=(16, 10))
sns.heatmap(season_episode_colors, annot=True, fmt='.1f', cmap='YlOrRd',
            cbar_kws={'label': 'Avg Number of Colors'},
            linewidths=0.5)
plt.title('🎨 Painting Complexity: Average # of Colors Used (Season × Episode)', 
         fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Episode Number', fontsize=12)
plt.ylabel('Season Number', fontsize=12)
plt.tight_layout()
plt.show()

print("What you see: Brighter = more colors used (more complex), Darker = fewer colors (simpler)")

In [ ]:
# HEATMAP #4: Top 12 Colors Across Episodes
# Shows usage of the 12 most popular colors episode by episode

top_12_colors = ['Titanium_White', 'Alizarin_Crimson', 'Van_Dyke_Brown', 
                 'Cadmium_Yellow', 'Yellow_Ochre', 'Phthalo_Blue',
                 'Bright_Red', 'Midnight_Black', 'Sap_Green',
                 'Indian_Yellow', 'Dark_Sienna', 'Prussian_Blue']

# Calculate average usage per episode
episode_top_colors = br_paint.groupby('episode')[top_12_colors].mean()

plt.figure(figsize=(14, 6))
sns.heatmap(episode_top_colors.T, annot=False, cmap='Blues',
            cbar_kws={'label': 'Frequency of Use'},
            linewidths=0.5)
plt.title('🎨 Top 12 Colors: Frequency Across All Episodes', 
         fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Episode Number', fontsize=12)
plt.ylabel('Color', fontsize=12)
plt.tight_layout()
plt.show()

print("What you see: Darker blue = used more often in that episode")

In [ ]:
# HEATMAP #5: Early vs Late Seasons Comparison
# Compare Bob's color preferences between early seasons (1-10) and late seasons (22-31)

early_seasons = br_paint[br_paint['season'] <= 10][color_columns].mean()
middle_seasons = br_paint[(br_paint['season'] > 10) & (br_paint['season'] <= 20)][color_columns].mean()
late_seasons = br_paint[br_paint['season'] > 20][color_columns].mean()

# Combine into one dataframe
era_comparison = pd.DataFrame({
    'Seasons 1-10 (Early)': early_seasons,
    'Seasons 11-20 (Middle)': middle_seasons,
    'Seasons 21-31 (Late)': late_seasons
})

plt.figure(figsize=(12, 8))
sns.heatmap(era_comparison, annot=True, fmt='.3f', cmap='RdYlGn',
            cbar_kws={'label': 'Frequency of Use'},
            linewidths=0.5)
plt.title('🎨 Evolution of Color Preferences: Early vs Middle vs Late Seasons', 
         fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Era', fontsize=12)
plt.ylabel('Color', fontsize=12)
plt.tight_layout()
plt.show()

print("What you see: Green = used more frequently, Red = used less frequently")
print("\nInsight: Did Bob change his color preferences over time?")

In [ ]:
# BONUS: CREATE YOUR OWN CUSTOM HEATMAP!
# Template for making any heatmap you want

# Example: Color usage by whether painting is from odd/even seasons
br_paint['season_type'] = br_paint['season'].apply(lambda x: 'Odd Season' if x % 2 == 1 else 'Even Season')

custom_heatmap = br_paint.groupby('season_type')[color_columns].mean()

plt.figure(figsize=(14, 4))
sns.heatmap(custom_heatmap, annot=True, fmt='.3f', cmap='PuBuGn',
            cbar_kws={'label': 'Frequency'},
            linewidths=1)
plt.title('🎨 Color Preferences: Odd vs Even Seasons', 
         fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n💡 TIP: Modify the groupby() parameter to create different heatmaps!")
print("Examples:")
print("  - groupby('season')  → colors per season")
print("  - groupby('episode') → colors per episode")
print("  - Or create your own custom grouping!")

# 📊 Heatmap Summary

## What You Can Visualize:
1. **Color Evolution** - How Bob's color choices changed over 31 seasons
2. **Color Correlations** - Which colors are always used together
3. **Complexity Trends** - Whether paintings got simpler or more complex over time
4. **Color Frequency** - Which colors dominated each episode
5. **Era Comparisons** - Early vs late career color preferences

## Heatmap Color Meanings:
- **Dark colors** = High frequency (common)
- **Light colors** = Low frequency (rare)
- **Red in correlation** = Strong positive relationship
- **Blue in correlation** = Strong negative relationship

## Why Heatmaps?
✓ Easy to spot patterns  
✓ Show changes over time  
✓ Compare multiple variables at once  
✓ Beautiful visualization


# Color Palette Clustering Analysis

Group paintings by their color combinations using K-means clustering on the binary color columns.

In [ ]:
# Extract color columns (binary indicators of color usage)
color_columns = [
    'Black_Gesso', 'Bright_Red', 'Burnt_Umber', 'Cadmium_Yellow', 
    'Dark_Sienna', 'Indian_Red', 'Indian_Yellow', 'Liquid_Black', 
    'Liquid_Clear', 'Midnight_Black', 'Phthalo_Blue', 'Phthalo_Green', 
    'Prussian_Blue', 'Sap_Green', 'Titanium_White', 'Van_Dyke_Brown', 
    'Yellow_Ochre', 'Alizarin_Crimson'
]

# Create feature matrix
X = br_paint[color_columns].copy()

print("Color features shape:", X.shape)
print("\nColor columns used:")
print(color_columns)

In [ ]:
# Standardize the features (important for K-means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled features shape:", X_scaled.shape)
print("Mean of scaled features (should be ~0):", X_scaled.mean())
print("Std of scaled features (should be ~1):", X_scaled.std())

In [ ]:
# Determine optimal number of clusters using Elbow Method
inertias = []
silhouette_scores = []
K_range = range(2, 11)

from sklearn.metrics import silhouette_score

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Plot Elbow curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Clusters (k)', fontsize=12)
ax1.set_ylabel('Inertia', fontsize=12)
ax1.set_title('Elbow Method for Optimal k', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(K_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
ax2.set_xlabel('Number of Clusters (k)', fontsize=12)
ax2.set_ylabel('Silhouette Score', fontsize=12)
ax2.set_title('Silhouette Score by Number of Clusters', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSilhouette Scores:")
for k, score in zip(K_range, silhouette_scores):
    print(f"k={k}: {score:.4f}")

In [ ]:
# Use k=4 clusters (good balance between interpretability and silhouette score)
optimal_k = 4
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
br_paint['color_cluster'] = kmeans_final.fit_predict(X_scaled)

print(f"\nClustering complete with k={optimal_k}")
print(f"Cluster distribution:")
print(br_paint['color_cluster'].value_counts().sort_index())
print(f"\nSilhouette Score: {silhouette_score(X_scaled, br_paint['color_cluster']):.4f}")

In [ ]:
# Analyze cluster characteristics
print("\n" + "="*80)
print("CLUSTER CHARACTERISTICS - Average Color Usage by Cluster")
print("="*80)

cluster_profiles = br_paint.groupby('color_cluster')[color_columns].mean()

# Display with better formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print(cluster_profiles.round(3))

# Create heatmap
plt.figure(figsize=(14, 6))
sns.heatmap(cluster_profiles, annot=True, fmt='.2f', cmap='YlOrRd', 
            cbar_kws={'label': 'Average Usage Frequency'},
            linewidths=0.5, linecolor='gray')
plt.title('Color Palette Profiles by Cluster', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Colors', fontsize=12)
plt.ylabel('Cluster', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Create cluster interpretations
print("\n" + "="*80)
print("CLUSTER INTERPRETATIONS")
print("="*80)

for cluster_id in range(optimal_k):
    cluster_data = br_paint[br_paint['color_cluster'] == cluster_id]
    print(f"\n🎨 CLUSTER {cluster_id} ({len(cluster_data)} paintings)")
    print("-" * 80)
    
    # Get most common colors in this cluster
    cluster_colors = cluster_profiles.loc[cluster_id]
    top_colors = cluster_colors.nlargest(8)
    
    print("Most frequent colors:")
    for color, freq in top_colors.items():
        print(f"  • {color}: {freq*100:.1f}%")
    
    print(f"\nAverage # of colors used: {cluster_data['num_colors'].mean():.2f}")
    print(f"Color count range: {cluster_data['num_colors'].min()}-{cluster_data['num_colors'].max()}")
    
    # Show a few example paintings
    print(f"\nExample paintings:")
    examples = cluster_data[['painting_title', 'season', 'episode', 'num_colors']].head(3)
    for idx, row in examples.iterrows():
        print(f"  - {row['painting_title']} (S{row['season']}E{row['episode']}, {row['num_colors']} colors)")

In [ ]:
# Visualize clusters using PCA (dimensionality reduction for 2D visualization)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"PCA Explained Variance Ratio: {pca.explained_variance_ratio_}")
print(f"Total Variance Explained: {sum(pca.explained_variance_ratio_):.4f}")

# Create visualization
plt.figure(figsize=(12, 8))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

for cluster_id in range(optimal_k):
    cluster_mask = br_paint['color_cluster'] == cluster_id
    plt.scatter(X_pca[cluster_mask, 0], X_pca[cluster_mask, 1], 
               label=f'Cluster {cluster_id}',
               s=100, alpha=0.6, color=colors[cluster_id], edgecolors='black', linewidth=0.5)

# Plot cluster centers
centers_pca = pca.transform(kmeans_final.cluster_centers_)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], 
           marker='*', s=1000, c='gold', edgecolors='black', linewidth=2,
           label='Cluster Centers', zorder=5)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
plt.title('Bob Ross Paintings - Color Palette Clusters (PCA Projection)', 
         fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Statistical summary of clusters
print("\n" + "="*80)
print("CLUSTER STATISTICS")
print("="*80)

summary_stats = br_paint.groupby('color_cluster').agg({
    'painting_title': 'count',
    'num_colors': ['mean', 'min', 'max', 'std'],
    'season': ['mean', 'min', 'max'],
    'episode': ['mean']
}).round(2)

summary_stats.columns = ['Count', 'Avg Colors', 'Min Colors', 'Max Colors', 'Std Colors',
                         'Avg Season', 'Min Season', 'Max Season', 'Avg Episode']
print(summary_stats)

print("\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)
print(f"\n✓ Successfully identified {optimal_k} distinct color palette clusters")
print(f"✓ Paintings distributed relatively evenly across clusters")
print(f"✓ Each cluster has unique color preferences and characteristics")
print(f"✓ Silhouette Score of {silhouette_score(X_scaled, br_paint['color_cluster']):.4f} indicates good cluster separation")

In [ ]:
# Visualize cluster composition by number of colors
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for cluster_id in range(optimal_k):
    cluster_data = br_paint[br_paint['color_cluster'] == cluster_id]
    axes[cluster_id].hist(cluster_data['num_colors'], bins=range(1, 16), 
                         color=colors[cluster_id], alpha=0.7, edgecolor='black')
    axes[cluster_id].set_xlabel('Number of Colors', fontsize=11)
    axes[cluster_id].set_ylabel('Frequency', fontsize=11)
    axes[cluster_id].set_title(f'Cluster {cluster_id}: Color Count Distribution\n({len(cluster_data)} paintings, Avg: {cluster_data["num_colors"].mean():.1f})', 
                              fontsize=12, fontweight='bold')
    axes[cluster_id].grid(True, alpha=0.3, axis='y')
    axes[cluster_id].set_xticks(range(1, 16))

plt.tight_layout()
plt.show()

In [ ]:
# Export clustered data for further analysis
output_df = br_paint[['painting_index', 'painting_title', 'season', 'episode', 
                       'num_colors', 'color_cluster']].copy()

print("\nSample of clustered paintings:")
print(output_df.head(10))

# Optionally save to CSV
# output_df.to_csv('bob_ross_clustered.csv', index=False)
# print("\n✓ Clustered data saved to 'bob_ross_clustered.csv'")